In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('Gold_Dec_26.csv')
df.head()

,Price,Close,High,Low,Open,Volume
0,Ticker,GC=F,GC=F,GC=F,GC=F,GC=F
1,Date,NaN,NaN,NaN,NaN,NaN
2,2000-08-30,273.8999938964844,273.8999938964844,273.8999938964844,273.8999938964844,0
3,2000-08-31,278.29998779296875,278.29998779296875,274.79998779296875,274.79998779296875,0
4,2000-09-01,277.0,277.0,277.0,277.0,0


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6563 entries, 0 to 6562
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Price   6563 non-null   str  
 1   Close   6562 non-null   str  
 2   High    6562 non-null   str  
 3   Low     6562 non-null   str  
 4   Open    6562 non-null   str  
 5   Volume  6562 non-null   str  
dtypes: str(6)
memory usage: 307.8 KB


Rows 0 and 1 (Ticker GC=F… and Date NaN…). Because of them, every column is read as text, not numbers. I should Load it by skipping the two first Rows 0 ans 1, I reload the file with `skiprows=[1, 2]` to skip these two lines

In [4]:
df = pd.read_csv('Gold_Dec_26.csv', skiprows=[1, 2])
df = df.rename(columns={'Price': 'Date'})
df['Date'] = pd.to_datetime(df['Date'])
df = df.set_index('Date')
df.head()

,Close,High,Low,Open,Volume
Date,,,,,
2000-08-30,273.899994,273.899994,273.899994,273.899994,0
2000-08-31,278.299988,278.299988,274.799988,274.799988,0
2000-09-01,277.000000,277.000000,277.000000,277.000000,0
2000-09-05,275.799988,275.799988,275.799988,275.799988,2
2000-09-06,274.200012,274.200012,274.200012,274.200012,0


In [5]:
df.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 6561 entries, 2000-08-30 to 2026-10-21
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Close   6561 non-null   float64
 1   High    6561 non-null   float64
 2   Low     6561 non-null   float64
 3   Open    6561 non-null   float64
 4   Volume  6561 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 307.5 KB


The data types are now correct, I need to print the first and last date to see the period of data. and also I need to check if the dates are in order because the Times Series depend on the times sequence. 

In [6]:
end_Date = df.index.max().date()
start_Date = df.index.min().date()
print(f'First Date : {start_Date}\nLast Date : {end_Date}')

First Date : 2000-08-30
Last Date : 2026-10-21


In [7]:
df.index.is_monotonic_increasing

True

After checking the chronogical order of dates, I should use `df.describe()` to see the minimum and maximum and the average of each column 

In [8]:
df.describe()

,Close,High,Low,Open,Volume
count,6561.000000,6561.000000,6561.000000,6561.000000,6561.000000
mean,1354.986831,1364.280994,1345.273266,1355.010456,51317.783112
std,919.961699,930.775494,908.476161,919.848664,92461.844497
min,255.100006,256.100006,255.000000,255.000000,0.000000
25%,660.299988,662.000000,658.900024,660.400024,29.000000
50%,1260.800049,1267.400024,1254.699951,1260.300049,199.000000
75%,1728.599976,1739.800049,1718.500000,1730.599976,87026.000000
max,5318.399902,5586.200195,5272.899902,5415.700195,570055.000000


There are no missing values because all 6561 rows have values in every column. From roughly **(min)** 255 USD in 2000 to a high close of **(max)** 5318.40. Which mean the gold increase in these period given (2000 - 2026), The standard deviation of the prices is very big (around 920). This is mainly because the price kept going up over the years. The data starts in 2000, but the assessment asks for the last ten years, so I will remove the older rows later.


# **Missing values, duplicates**

In [9]:
missing_values = df.isnull().sum()
missing_values

Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64

There is no missing values, let's check if There is any duplicates dates

In [10]:
duplicates_values = df.duplicated().sum()
print(duplicates_values)

41


In [11]:
duplicates_dates = df.index.duplicated().sum()
duplicates_dates

np.int64(0)

At first I used `df.duplicated()`, but that checks if whole rows have the same values, not if the same date appears twice. It found 41 rows, but these were different days that just had the same prices, so they are real days and I should not delete them.

In [12]:
check_neg_values = df[['Close', 'High', 'Low', 'Open']] <= 0
check_neg_values.any(axis = 1).sum()

np.int64(0)

In [13]:
(df['High'] < df['Low']).sum()

np.int64(1)

In [14]:
((df['Open'] > df['High']) | (df['Open'] < df['Low'])).sum()

np.int64(56)

In [15]:
((df['Close'] > df['High']) | (df['Close'] < df['Low'])).sum()

np.int64(430)

In [16]:
((df['Volume'] == 0).sum())

np.int64(418)

### Last 10 years

In [17]:
# currentDate = pd.Timestamp.now()
# lastTenYear = pd.DateOffset(years = 10)

# # currentDate
# ten_years_ago = (currentDate - lastTenYear).date()
# print(ten_years_ago)
start_date = '2016-10-21'
end_date = '2026-10-20'



In [18]:
df_10_years = df.loc[start_date: end_date]
df_10_years

,Close,High,Low,Open,Volume
Date,,,,,
2016-10-21,1265.900024,1265.900024,1265.900024,1265.900024,224
2016-10-24,1262.000000,1269.199951,1261.800049,1269.199951,42
2016-10-25,1271.900024,1274.099976,1264.099976,1264.099976,61
2016-10-26,1264.800049,1274.000000,1264.800049,1274.000000,279
2016-10-27,1267.900024,1269.400024,1267.900024,1269.400024,338
...,...,...,...,...,...
2026-10-14,4107.899902,4193.500000,4095.000000,4184.200195,191279
2026-10-15,4170.100098,4197.700195,4120.899902,4128.299805,159510
2026-10-16,4181.100098,4220.399902,4167.500000,4192.100098,137939


In [19]:
print(len(df_10_years))

2513


In [20]:
df_10_years.describe()

,Close,High,Low,Open,Volume
count,2513.000000,2513.000000,2513.000000,2513.000000,2513.000000
mean,2100.841743,2117.194230,2083.653999,2100.651532,127860.603263
std,975.148469,990.480386,958.777356,974.970352,110035.380647
min,1127.800049,1132.800049,1123.900024,1126.900024,0.000000
25%,1335.400024,1340.500000,1328.500000,1335.000000,330.000000
50%,1815.900024,1827.300049,1804.900024,1816.699951,147947.000000
75%,2327.000000,2346.100098,2310.899902,2329.899902,205084.000000
max,5318.399902,5586.200195,5272.899902,5415.700195,570055.000000


In [21]:
df.describe()

,Close,High,Low,Open,Volume
count,6561.000000,6561.000000,6561.000000,6561.000000,6561.000000
mean,1354.986831,1364.280994,1345.273266,1355.010456,51317.783112
std,919.961699,930.775494,908.476161,919.848664,92461.844497
min,255.100006,256.100006,255.000000,255.000000,0.000000
25%,660.299988,662.000000,658.900024,660.400024,29.000000
50%,1260.800049,1267.400024,1254.699951,1260.300049,199.000000
75%,1728.599976,1739.800049,1718.500000,1730.599976,87026.000000
max,5318.399902,5586.200195,5272.899902,5415.700195,570055.000000


### Compare `df` & `df_10_years`:
> - Gold prices are much higher in this period: the average close is about 2,100 USD compared with about 1,355 USD for the full data, and the lowest close is about 1,128 USD instead of 255 USD.
> - The maximum values did not change (highest close 5,318.40 and highest High 5,586.20), so the record prices happened in the last ten years.

`df_10_years.describe()` showed that 25% of day in the last ten years have a fewer about 330, while the median (50%) is about 148000, which is a big difference.

In [22]:
(df_10_years['High'] < df_10_years['Low']).sum()

np.int64(0)

Let's if the full data changed compar with the data of the last 10 year.

In [23]:
# Remaining checks on the ten-year data
open_outside = ((df_10_years['Open'] > df_10_years['High']) | (df_10_years['Open'] < df_10_years['Low'])).sum()
close_outside = ((df_10_years['Close'] > df_10_years['High']) | (df_10_years['Close'] < df_10_years['Low'])).sum()
zero_volume = (df_10_years['Volume'] == 0).sum()

print('Open outside High-Low range:', open_outside)
print('Close outside High-Low range:', close_outside)
print('Days with zero volume:', zero_volume)

Open outside High-Low range: 0
Close outside High-Low range: 0
Days with zero volume: 36


In the last ten years, the Open and Close are always between the High and the Low, and High is never lower than Low. So all the price problems I found earlier were only in the old years and were removed by the 10 year.

In [24]:

low_volume = df_10_years['Volume'] < 1000
print('Days with volume below 1,000:', low_volume.sum(), 'out of', len(df_10_years))

low_volume.groupby(df_10_years.index.year).sum()

Days with volume below 1,000: 810 out of 2513


Date
2016     41
2017    223
2018    228
2019    224
2020     86
2021      0
2022      0
2023      1
2024      0
2025      5
2026      2
Name: Volume, dtype: int64

> From 2016 to early 2020, almost every day has a tiny volume, and from 2021 onwards almost none do
>  Volume isn’t consistent across the ten years, so we won’t use it as a clustering feature. Otherwise the clusters would mostly separate “before 2020” from “after 2020” instead of different market conditions.

After the data cleaned, I want to name my own data to **gold** and using `copy` to make new data indepent from `df_10_years`

In [25]:
gold = df_10_years.copy()

In [26]:
gold.shape

(2513, 5)

In [27]:
gold.columns 

Index(['Close', 'High', 'Low', 'Open', 'Volume'], dtype='str')

In [28]:
gold.index.date.min()

datetime.date(2016, 10, 21)

In [29]:
gold.index.date.max()

datetime.date(2026, 10, 20)

# The Golden Rule

As the course recommend :
```
> Logging: In the case where the compound rate in the trend is not due to a price index (i.e., the series is not measured in a currency), logging can helpto linearizea series with an exponential trend (recall that log(exp(x)) = x). It does not remove an eventual trend whatsoever, unlike deflation.

> We use Close, because gold futures have no Adjusted Close.
```



In [30]:
# Daily log return: relative change in the closing price from the previous day
gold['Log_Return'] = np.log(gold['Close'] / gold['Close'].shift(1))
gold[['Close', 'Log_Return']].head()

,Close,Log_Return
Date,,
2016-10-21,1265.900024,NaN
2016-10-24,1262.000000,-0.003086
2016-10-25,1271.900024,0.007814
2016-10-26,1264.800049,-0.005598
2016-10-27,1267.900024,0.002448


As we can see **NaN** in the first row, because the first day has no previous day to compare with
> I calculate the daily price range as a share of the Close price. It shows how much the price moved during the day, which the return does not show. Dividing by Close makes the range comparable across the whole period.

In [31]:
gold['Daily_Range'] = (gold['High'] - gold['Low']) / gold['Close']
gold[['High', 'Low', 'Close', 'Daily_Range']].head()

,High,Low,Close,Daily_Range
Date,,,,
2016-10-21,1265.900024,1265.900024,1265.900024,0.000000
2016-10-24,1269.199951,1261.800049,1262.000000,0.005864
2016-10-25,1274.099976,1264.099976,1271.900024,0.007862
2016-10-26,1274.000000,1264.800049,1264.800049,0.007274
2016-10-27,1269.400024,1267.900024,1267.900024,0.001183
